In [ ]:
# 1. read HomologTraningSet.CSV
# 2. Create new dataframe with the columns (group,family,protein_name,species,uniprot_id,pdb_id,binding_site_type,key_binding_residues,notes)
#    from the original CSV
# 3. Next need to run af3, chai1, boltz2 for each protein once
# 4. The protein files immediately need to be parsed and moved, so it will not disturb future outputs
# 5. The parsed .cif files need to be added to a directory
# 6. The paths of the files need to be added to the new dataframe along with the generation model name and sample number
# If running on tacc may be able to run all concurrently, on Hopfog only have 2 gpus on a server, need to balance processes between the
# 7. Need to modify the binding residues to be the indices (starting at 0) they need to match the sequence length
#    new column with correct_final_scores will be an array of 0's except at known binding residues with a 
# 8. Save CSV

In [1]:
%pip install biopython

  Using cached biopython-1.87-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached numpy-2.4.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached biopython-1.87-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
Using cached numpy-2.4.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from Bio.PDB import PDBList
from Bio import SeqIO

In [3]:
%pip install pandas

  Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

In [5]:
training_set_df = pd.read_csv("/stor/home/sje779/Work/FinalProject/model_training/HomologTrainingSet.csv")
training_set_df.head()

,group,family,protein_name,species,uniprot_id,pdb_id,binding_site_type,key_binding_residues,notes
0,1,Serine Protease,Trypsin,Bos taurus,P00760,1TRN,catalytic triad,H57;D102;S195,canonical chymotrypsin numbering
1,1,Serine Protease,Chymotrypsin A,Bos taurus,P00766,4CHA,catalytic triad,H57;D102;S195,high sequence/structural homology
2,1,Serine Protease,Elastase,Sus scrofa,P00772,1ELA,catalytic triad,H57;D102;S195,substrate pocket differs
3,1,Serine Protease,Thrombin,Homo sapiens,P00734,1PPB,catalytic triad,H57;D102;S195,coagulation protease
4,1,Serine Protease,Factor Xa,Homo sapiens,P00742,2P16,catalytic triad,H57;D102;S195,drug target protease


In [6]:
from pathlib import Path

In [24]:
CIF_DIR = "/stor/home/sje779/Work/FinalProject/model_training/Models"


def extract_sequence(pdb_id: str) -> str | None:
    cif_path = Path(CIF_DIR).joinpath(f"{pdb_id.lower()}.cif")
    if not cif_path.exists():
        print(f"[SEQ ERROR] CIF not found for {pdb_id}")
        return None
    try:
        records = list(SeqIO.parse(str(cif_path), "cif-seqres"))
        if not records:
            records = list(SeqIO.parse(str(cif_path), "cif-atom"))
        if records:
            return str(records[0].seq)
        return None
    except Exception as e:
        print(f"[SEQ ERROR] {pdb_id}: {e}")
        return None



pdbl = PDBList()

sequences = []
file_paths = []

for pdb_id in training_set_df["pdb_id"]:
    pdb_id = pdb_id.lower()
    
    cif_path = pdbl.retrieve_pdb_file(
        pdb_id,
        file_format="mmCif",
        pdir=str(CIF_DIR),
        overwrite=False
    )
    seq = extract_sequence(pdb_id)
    sequences.append(seq)
    file_paths.append(cif_path)

training_set_df["sequence"] = sequences
training_set_df["file_path"] = file_paths

Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1trn.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/4cha.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1ela.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1ppb.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/2p16.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/2hnp.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1l8k.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/2shp.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/3ps5.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1ygu.cif' 
Structure exists: '/stor/home/sje779/Work/FinalProject/model_training/Models/1hck.cif' 
Structure exists: '/stor/home/sj

In [11]:
training_set_df.head()

,group,family,protein_name,species,uniprot_id,pdb_id,binding_site_type,key_binding_residues,notes,sequence,file_path
0,1,Serine Protease,Trypsin,Bos taurus,P00760,1TRN,catalytic triad,H57;D102;S195,canonical chymotrypsin numbering,IVGGYNCEENSVPYQVSLNSGYHFCGGSLINEQWVVSAGHCYKSRI...,/stor/home/sje779/Work/FinalProject/model_trai...
1,1,Serine Protease,Chymotrypsin A,Bos taurus,P00766,4CHA,catalytic triad,H57;D102;S195,high sequence/structural homology,CGVPAIQPVLSGL|IVNGEEAVPGSWPWQVSLQDKTGFHFCGGSLI...,/stor/home/sje779/Work/FinalProject/model_trai...
2,1,Serine Protease,Elastase,Sus scrofa,P00772,1ELA,catalytic triad,H57;D102;S195,substrate pocket differs,VVGGTEAQRNSWPSQISLQYRSGSSWAHTCGGTLIRQNWVMTAAHC...,/stor/home/sje779/Work/FinalProject/model_trai...
3,1,Serine Protease,Thrombin,Homo sapiens,P00734,1PPB,catalytic triad,H57;D102;S195,coagulation protease,TFGSGEADCGLRPLFEKKSLEDKTERELLESYIDGR|IVEGSDAEI...,/stor/home/sje779/Work/FinalProject/model_trai...
4,1,Serine Protease,Factor Xa,Homo sapiens,P00742,2P16,catalytic triad,H57;D102;S195,drug target protease,IVGGQECKDGECPWQALLINEENEGFCGGTILSEFYILTAAHCLYQ...,/stor/home/sje779/Work/FinalProject/model_trai...


In [14]:
import json

In [15]:
%pip install pyyaml

  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (801 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import yaml

In [25]:
def generate_af3_json(pdb_id, sequence, work_dir):
    root = {"name": pdb_id,
            "modelSeeds": [1],
            "dialect": "alphafold3",
            "version": 4,
            "sequences": [
                {
                "protein": {
                    "id": "A",
                    "sequence": sequence
                    }
                }
            ]
        }
    
    file_path = Path(work_dir).joinpath(f"{pdb_id}.json")
    with open(file_path, "w") as out:
        json.dump(root, out, indent=4)


def generate_boltz2_yaml(pdb_id, sequence, work_dir):
    root = {
        "version": 1,
        "inference": {
            "seed": 0,
            "num_seeds": 1
        },
        "sequences":[{
            "protein": {
                "id": "A",
                "sequence": sequence
            }
        }]
    }

    file_path = Path(work_dir).joinpath(f"{pdb_id}.yaml")
    with open(file_path, "w") as out:
        yaml.dump(root, out, default_flow_style=False, sort_keys=False)


def generate_chai1_fasta(pdb_id, sequence, work_dir):
    file_path = Path(work_dir).joinpath(f"{pdb_id}.fasta")
    with open(file_path, "w") as out:
        out.write(f">Protein|name={pdb_id}\n")
        out.write(sequence)


AF3_INPUT_PATH = "/stor/home/sje779/Work/FinalProject/model_training/af3_input_json"
BOLTZ2_INPUT_PATH = "/stor/home/sje779/Work/FinalProject/model_training/boltz2_input_yaml"
CHAI1_INPUT_PATH = "/stor/home/sje779/Work/FinalProject/model_training/chai1_input_fasta"

for path in [AF3_INPUT_PATH, BOLTZ2_INPUT_PATH, CHAI1_INPUT_PATH]:
    Path(path).mkdir(parents=True, exist_ok=True)

for pdb_id, sequence in zip(training_set_df["pdb_id"], training_set_df["sequence"]):
    generate_af3_json(pdb_id, sequence, AF3_INPUT_PATH)
    generate_boltz2_yaml(pdb_id, sequence, BOLTZ2_INPUT_PATH)
    generate_chai1_fasta(pdb_id, sequence, CHAI1_INPUT_PATH)
